In [1]:
using Revise
using GeometryBasics: Vec3f, Point3f
using LinearAlgebra
using GLMakie
using StaticArrays
using VMRobotControl
using VMRobotControl.Splines: CubicSpline
using DifferentialEquations
using MeshIO
include("../functions.jl")

circle_center_tangent_to_lines (generic function with 1 method)

In [9]:
using FileIO, UUIDs
try
    FileIO.add_format(format"DAE", (), ".dae", [:DigitalAssetExchangeFormatIO => UUID("43182933-f65b-495a-9e05-4d939cea427d")])
catch
end

cfg = URDFParserConfig(;suppress_warnings=true) # This is just to hide warnings about unsupported URDF features
module_path = joinpath(splitpath(splitdir(pathof(VMRobotControl))[1])[1:end-1])
robot = parseURDF(joinpath(module_path, "URDFs/sr_description/sr_hand_vm_compatible.urdf"), cfg)

24DOF Mechanism{Float64} "shadowhand_motor" with 33 frames, 32 joints, 55 coordinates, 82 components

In [10]:
add_gravity_compensation!(robot, VMRobotControl.DEFAULT_GRAVITY)

joint_limits = cfg.joint_limits

for joint_id in keys(joints(robot))
    limits = joint_limits[joint_id]
    isnothing(limits) && continue
    add_coordinate!(robot, JointSubspace(joint_id);  id="$(joint_id)_coord")
    @assert ~isnothing(limits.lower) && ~isnothing(limits.upper)
    add_deadzone_springs!(robot, 0.01, (limits.lower+0.0, limits.upper-0.0), "$(joint_id)_coord")
    add_component!(robot, LinearDamper(0.0001, "$(joint_id)_coord"); id="$(joint_id)_damper")
end

In [28]:
# SPRINGS AND DAMPERS
vms = VirtualMechanismSystem("myShadowVMS", robot)
root = root_frame(vms.robot)
add_coordinate!(vms, FramePoint(".robot.$root", SVector(0.0, -0.02, 0.3));        id="Target position")
add_coordinate!(vms, CoordDifference(".robot.rh_fftip_mass_coord", "Target position"); id="Position error");

K = SMatrix{3, 3}(0.01, 0., 0., 0., 0.01, 0., 0., 0., 0.01)
add_component!(vms, LinearSpring(K, "Position error");  id="Linear Spring")
D = SMatrix{3, 3}(0.001, 0., 0., 0., 0.001, 0., 0., 0., 0.001)
add_component!(vms, LinearDamper(D, "Position error");  id="Linear Damper")

add_component!(vms, LinearSpring(0.1, ".robot.rh_LFJ5_coord"); id = "lf j5 angular spring")

# SIMULATION 
tspan = (0., 10.)
vms_compiled = compile(vms)
q = (zero_q(vms_compiled.robot), Float64[]) # Robot joint angle, vm joint angles
q̇ = (zero_q̇(vms_compiled.robot), Float64[]) # Robot joint velocity, vm joint velocities
g = VMRobotControl.DEFAULT_GRAVITY
dcache = new_dynamics_cache(vms_compiled)
prob = get_ode_problem(dcache, g, q, q̇, tspan)
sol = solve(prob, Rosenbrock23(autodiff=false), progress=true; maxiters=1e6, abstol=1e-3, reltol=1e-3);

ODE   0%|█                                              |  ETA: N/A


In [29]:
# ANIMATION
fig = Figure(size = (720, 720), figure_padding=0)
display(fig)
ls = LScene(fig[1, 1]; show_axis=false)
cam = cam3d!(ls, camera=:perspective, center=false)
cam.lookat[] = [-0.6, 0.0, 0.0]
cam.eyeposition[] = [0.87, 0.04, 0.41]

plotting_t = Observable(0.0)
plotting_kcache = Observable(new_kinematics_cache(compile(robot)))
robotvisualize!(ls, plotting_kcache;)

animate_robot_odesolution(fig, sol, plotting_kcache, "test.mp4"; t=plotting_t);